# NB04: District Segmentation

## Welfare Scheme Participation Analysis

Cluster districts into actionable segments using KMeans + PCA, based on scheme gaps and demographics.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import MASTER_DATA
from src.feature_engineer import build_features
from src.segmentation import (
    prepare_clustering_data, find_optimal_k, fit_kmeans,
    run_pca, save_cluster_assignments, SEGMENTATION_FEATURES
)
from src.visualization import (
    setup_plot_style, save_figure, plot_elbow_curve,
    plot_cluster_pca, plot_cluster_profiles
)

setup_plot_style()
master = pd.read_csv(MASTER_DATA)
df = build_features(master)
print(f'Dataset: {df.shape[0]} districts')

## 1. Prepare Clustering Data

In [ ]:
df_valid, X_scaled, scaler = prepare_clustering_data(df)
print(f'Valid districts for clustering: {len(df_valid)}')
print(f'Features: {SEGMENTATION_FEATURES}')

## 2. Optimal K Selection

In [ ]:
k_results = find_optimal_k(X_scaled)
print(k_results.to_string(index=False))

fig = plot_elbow_curve(k_results, save_name='elbow_curve')

In [ ]:
best_k = k_results.loc[k_results['silhouette'].idxmax(), 'k']
print(f'Optimal k (max silhouette): {int(best_k)}')

## 3. KMeans Clustering

In [ ]:
km, labels, cluster_profiles, df_clustered = fit_kmeans(
    X_scaled, df_valid, k=int(best_k)
)
print(f'Cluster sizes:')
print(df_clustered['cluster_label'].value_counts().to_string())

## 4. Cluster Profiles

In [ ]:
profile_features = [c for c in SEGMENTATION_FEATURES if c in cluster_profiles.columns]
print(cluster_profiles[profile_features + ['n_districts']].round(1).to_string())

In [ ]:
fig = plot_cluster_profiles(
    cluster_profiles, profile_features,
    'District Cluster Profiles',
    save_name='cluster_profiles'
)

## 5. PCA Visualization

In [ ]:
pca, components, explained = run_pca(X_scaled)
print(f'PCA explained variance: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}, Total={explained.sum():.1%}')

In [ ]:
fig = plot_cluster_pca(
    components, labels,
    f'KMeans Clusters (k={int(best_k)}) on PCA Space',
    save_name='cluster_pca'
)

## 6. Save Cluster Assignments

In [ ]:
df_clustered = save_cluster_assignments(df_clustered)
print(f'Saved cluster assignments for {len(df_clustered)} districts')